# Session 15: Causal Inference and A/B Testing

**Applied Machine Learning Using Python**  
**Duration**: 4 Hours (TL15)  
**Level**: 🟡 Intermediate

---

## 📋 Objectives
1. Prove Causation through A/B Testing.
2. Determine Statistical Significance (p-values).
3. Combat Bias with Propensity Score Matching (PSM).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Libraries Loaded Successfully.")

---
## §1 — The A/B Test (Randomized Controlled Trial)
Imagine you are a Data Scientist at Netflix. You want to test if a **Red Play Button** (Treatment/Group B) causes more users to click than the **Current White Play Button** (Control/Group A).

You split 2000 users exactly 50/50 randomly. Because of the random split, there is NO bias.

In [ ]:
# Simulate the test
np.random.seed(42)
# Array of 1000 users. '1' means they clicked. '0' means they ignored it.
group_a = np.random.binomial(n=1, p=0.08, size=1000) # 8% base conversion rate
group_b = np.random.binomial(n=1, p=0.11, size=1000) # 11% conversion rate for Red Button

print(f"Conversion Rate A: {group_a.mean() * 100:.1f}%")
print(f"Conversion Rate B: {group_b.mean() * 100:.1f}%")
print("\nGroup B got more clicks! But wait... is it just random luck? We must run a T-Test.")

In [ ]:
# Run an Independent T-Test
t_stat, p_value = stats.ttest_ind(group_b, group_a)

print(f"P-Value: {p_value:.4f}")

if p_value < 0.05:
    print("HURRAY! The p-value is less than 0.05. The result is Statistically Significant.")
    print("Decision: We confidently roll out the Red Button to everyone.")
else:
    print("FAIL! The p-value is greater than 0.05. The difference was likely just random luck.")
    print("Decision: We throw away the Red Button and keep investigating.")

---
## §2 — Propensity Score Matching (Observational Data)
Sometimes, A/B testing is illegal or impossible. 

Imagine measuring if taking a "Data Science Bootcamp" causes higher salaries. You cannot force half a city to randomly quit their jobs to take the bootcamp. You must rely on *Observational Data* of people who *chose* to take it.

**The Bias (Confounding Variables):** People who take bootcamps are naturally highly-motivated and eager. Highly-motivated people make more money anyway! Did the Bootcamp cause the raise, or did Motivation cause the raise?

Let's simulate this extremely biased data.

In [ ]:
np.random.seed(42)
n = 1000
motivation_level = np.random.normal(50, 15, n) # Confounding Variable

# People with high motivation are much more likely to take the bootcamp (Treatment = 1)
treatment_probability = 1 / (1 + np.exp(-(-5 + 0.1 * motivation_level)))
bootcamp_taken = np.random.binomial(1, treatment_probability)

# The true impact of the bootcamp is only +$5000.
# BUT high motivation also naturally adds $1000 per motivation point.
salary = 40000 + (motivation_level * 1000) + (5000 * bootcamp_taken) + np.random.normal(0, 5000, n)

df_obs = pd.DataFrame({'Motivation': motivation_level, 'Took_Bootcamp': bootcamp_taken, 'Salary': salary})

print("BIASED OBSERVATIONAL RESULTS:")
print(f"Avg Salary (No Bootcamp): ${df_obs[df_obs['Took_Bootcamp']==0]['Salary'].mean():,.2f}")
print(f"Avg Salary (Bootcamp):    ${df_obs[df_obs['Took_Bootcamp']==1]['Salary'].mean():,.2f}")

diff = df_obs[df_obs['Took_Bootcamp']==1]['Salary'].mean() - df_obs[df_obs['Took_Bootcamp']==0]['Salary'].mean()
print(f"Naive (Wrong) Conclusion: The bootcamp increases your salary by ${diff:,.2f}.")
print("This is wildly false! The true impact was only $5000. Motivation skewed the results!")

### Step 2a: Train the PSM Logistic Regression Model
We use a Logistic Regression ML Model to calculate exactly how likely every person was to take the bootcamp based on their `Motivation`.

In [ ]:
from sklearn.linear_model import LogisticRegression

ps_model = LogisticRegression()
ps_model.fit(df_obs[['Motivation']], df_obs['Took_Bootcamp'])

# Assign Propensity Scores (The probability of taking the bootcamp)
df_obs['Propensity_Score'] = ps_model.predict_proba(df_obs[['Motivation']])[:, 1]

plt.figure(figsize=(8,4))
sns.histplot(data=df_obs, x='Propensity_Score', hue='Took_Bootcamp', bins=30)
plt.title("Distribution of Propensity Scores")
plt.show()
print("Now that we have scores, we must match them!")

### Step 2b: Matching with K-Nearest Neighbors
We find the 1 closest person from the "Did Not Take Bootcamp" group to perfectly match with someone who "Took Bootcamp".

In [ ]:
from sklearn.neighbors import NearestNeighbors

treatment = df_obs[df_obs['Took_Bootcamp'] == 1]
control = df_obs[df_obs['Took_Bootcamp'] == 0]

knn = NearestNeighbors(n_neighbors=1)
knn.fit(control[['Propensity_Score']])

distances, indices = knn.kneighbors(treatment[['Propensity_Score']])
matched_control = control.iloc[indices.flatten()]

# The balanced, unbiased dataset mimicking an A/B Test:
matched_df = pd.concat([treatment, matched_control])

true_diff = matched_df[matched_df['Took_Bootcamp']==1]['Salary'].mean() - matched_df[matched_df['Took_Bootcamp']==0]['Salary'].mean()
print(f"✅ CORRECT Conclusion: After applying PSM, the bootcamp only increases your salary by ${true_diff:,.2f}.")
print("We successfully extracted the True Causal Impact by eliminating the Confounding Variable!")